# E-Commerce Lakehouse: Silver Products

## Standardize product columns and data types

Correct the source spelling of the name and description length
columns. Convert product measurements and counts to integers.

Preserve missing values rather than replacing unknown measurements
with zero. Keep the original Bronze table unchanged.

In [0]:
from pyspark.sql.functions import col

bronze_products_df = spark.table(
    "workspace.ecommerce.bronze_products"
)

silver_products_df = (
    bronze_products_df
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )
)

numeric_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column_name in numeric_columns:
    silver_products_df = silver_products_df.withColumn(
        column_name,
        col(column_name).cast("int")
    )

silver_products_df.printSchema()
display(silver_products_df.limit(5))

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-10-17T20:19:41.733Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-10-17T20:19:41.733Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-10-17T20:19:41.733Z
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-10-17T20:19:41.733Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-10-17T20:19:41.733Z


## Validate Silver products

Verify row counts, required product IDs, and product ID uniqueness.
Compare numeric null counts before and after conversion.
Measure missing categories and flag negative numeric values
without filling unknown values or deleting records.

In [0]:
from pyspark.sql.functions import trim
from functools import reduce

print("Bronze product rows:", bronze_products_df.count())
print("Silver product rows:", silver_products_df.count())

missing_ids = silver_products_df.filter(
    col("product_id").isNull()
    | (trim(col("product_id")) == "")
).count()

duplicate_ids = (
    silver_products_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Missing or blank product IDs:", missing_ids)
print("Duplicate product IDs:", duplicate_ids)

# Map renamed Silver fields to their original Bronze names.
source_names = {
    "product_name_length": "product_name_lenght",
    "product_description_length": "product_description_lenght"
}

for column_name in numeric_columns:
    source_name = source_names.get(column_name, column_name)

    bronze_nulls = bronze_products_df.filter(
        col(source_name).isNull()
    ).count()

    silver_nulls = silver_products_df.filter(
        col(column_name).isNull()
    ).count()

    print(
        f"{column_name}: "
        f"Bronze nulls={bronze_nulls}, Silver nulls={silver_nulls}"
    )

missing_categories = silver_products_df.filter(
    col("product_category_name").isNull()
    | (trim(col("product_category_name")) == "")
).count()

print("Missing or blank product categories:", missing_categories)

# Count rows with at least one negative numeric value.
negative_condition = reduce(
    lambda left, right: left | right,
    [col(column_name) < 0 for column_name in numeric_columns]
)

print(
    "Products with negative numeric values:",
    silver_products_df.filter(negative_condition).count()
)

Bronze product rows: 32951
Silver product rows: 32951
Missing or blank product IDs: 0
Duplicate product IDs: 0
product_name_length: Bronze nulls=610, Silver nulls=610
product_description_length: Bronze nulls=610, Silver nulls=610
product_photos_qty: Bronze nulls=610, Silver nulls=610
product_weight_g: Bronze nulls=2, Silver nulls=2
product_length_cm: Bronze nulls=2, Silver nulls=2
product_height_cm: Bronze nulls=2, Silver nulls=2
product_width_cm: Bronze nulls=2, Silver nulls=2
Missing or blank product categories: 610
Products with negative numeric values: 0


## Validate relationships and refresh Silver products

Check product IDs, expected row count, and links from order items.
Replace the Silver products table only after these checks pass.
Verify the saved row count while preserving missing source attributes.

In [0]:
from pyspark.sql.functions import col, trim

silver_order_items_df = spark.table(
    "workspace.ecommerce.silver_order_items"
)

unmatched_count = silver_order_items_df.join(
    silver_products_df.select("product_id"),
    on="product_id",
    how="left_anti"
).count()

row_count = silver_products_df.count()

missing_id_count = silver_products_df.filter(
    col("product_id").isNull()
    | (trim(col("product_id")) == "")
).count()

duplicate_id_count = (
    silver_products_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Order items without a matching product:", unmatched_count)
print("Silver product rows:", row_count)
print("Missing or blank product IDs:", missing_id_count)
print("Duplicate product IDs:", duplicate_id_count)

if (
    unmatched_count != 0
    or row_count != 32951
    or missing_id_count != 0
    or duplicate_id_count != 0
):
    raise ValueError(
        "Silver product validation failed. "
        "Stopped before replacing the table."
    )

table_name = "workspace.ecommerce.silver_products"

(
    silver_products_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_products_df = spark.table(table_name)
saved_count = saved_products_df.count()

if saved_count != row_count:
    raise ValueError("Saved Silver product count does not match.")

print("Silver products refreshed:", saved_count)

Order items without a matching product: 0
Silver product rows: 32951
Missing or blank product IDs: 0
Duplicate product IDs: 0
Silver products refreshed: 32951
